# Assignment 3 - Network biology

### Setup and Required Libraries

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import seaborn as sns
from itertools import product
from collections import Counter, defaultdict
import warnings
import copy
import itertools
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

### Boolean Network Model Setup

In [ ]:
class BooleanNetwork:
    def __init__(self, node_names):
        self.nodes = {name: 0 for name in node_names}
        self.rules = {}
        self.history = []
        self.graph = nx.DiGraph()  # NetworkX graph for visualization

        # Add nodes to NetworkX graph
        self.graph.add_nodes_from(node_names)

    def add_rule(self, target_node, rule_function, rule_description=""):
        """
        Add Boolean rule for a node

        Args:
            target_node: Node to update
            rule_function: Function that takes current state dict and returns True/False
            rule_description: Human-readable description
        """
        self.rules[target_node] = {
            'function': rule_function,
            'description': rule_description
        }

    def set_state(self, **kwargs):
        """Set states of specific nodes"""
        for node, value in kwargs.items():
            if node in self.nodes:
                self.nodes[node] = int(bool(value))

    def get_state_vector(self):
        """Get current state as list in sorted order"""
        return [self.nodes[node] for node in sorted(self.nodes.keys())]

    def update_synchronous(self):
        """Update all nodes simultaneously"""
        new_state = {}
        for node in self.nodes:
            if node in self.rules:
                new_state[node] = int(self.rules[node]['function'](self.nodes))
            else:
                new_state[node] = self.nodes[node]  # No rule = no change

        self.nodes = new_state
        self.history.append(self.get_state_vector())

    def simulate(self, steps=10, record_history=True, to_print=True):
        """Run simulation"""
        if record_history:
            self.history = [self.get_state_vector()]

        for step in range(steps):
            self.update_synchronous()

            # Check for steady state
            if len(self.history) >= 2 and self.history[-1] == self.history[-2]:
                if to_print:
                    print(f"   Reached steady state after {step+1} steps")
                break
        else:
            if to_print:
                print("   Steady state wasn't reached")

        return np.array(self.history)

In [ ]:
nodes = ['DNA_damage', 'p53', 'MYC', 'CDK2', 'MDM2', 'p21', 'Growth', 'Death']
network = BooleanNetwork(nodes)

# Define Boolean rules (based on real biology, simplified)
network.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
network.add_rule('p21', lambda s: s['p53'], "p21 = p53")
network.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
network.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
network.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
network.add_rule('p53', lambda s: s['DNA_damage'] and not s['MDM2'], "p53 = DNA_damage AND (NOT MDM2)")
network.add_rule('Growth',lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
network.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")

print("Rules:")
for node, rule_info in network.rules.items():
    print(f" {rule_info['description']}")

Rules:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)


### Mutated Networks

In [ ]:
# 1. Mutation A: p53 Knockout (Loss of tumor suppressor): 
network_mut_a = copy.deepcopy(network)
network_mut_a.add_rule('p53',lambda s:False, "p53 = BROKEN (always OFF)")

# 2. Mutation B: MYC Amplification (Oncogene overexpression)
network_mut_b = copy.deepcopy(network)
network_mut_b.add_rule('MYC', lambda s: True, "MYC = AMPLIFIED (always ON)")

# 3. Mutation C: MDM2 Overexpression (p53 pathway disruption): 
network_mut_c = copy.deepcopy(network)
network_mut_c.add_rule('MDM2', lambda s: True, "MDM2 = OVEREXPRESSED (always ON)")

# 4. Mutation D: p21 Knockout
network_mut_d = copy.deepcopy(network)
network_mut_d.add_rule('p21',lambda s: False,"p21 = BROKEN (always OFF)")

network_list = [network_mut_a,network_mut_b, network_mut_c, network_mut_d ]

print("Rules:")
for i, net in enumerate(network_list):
    print(f"\nMutation {i + 1}:")
    for node, rule_info in net.rules.items():
        print(f" {rule_info['description']}")



Rules:

Mutation 1:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = BROKEN (always OFF)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)

Mutation 2:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = AMPLIFIED (always ON)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)

Mutation 3:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = OVEREXPRESSED (always ON)
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)

Mutation 4:
 DNA_damage = INPUT (constant)
 p21 = BROKEN (always OFF)
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p

### Scenario Analysis

In [ ]:
networks = {
    'Normal Network': network,
    'Mutation A (p53 Knockout)': network_mut_a,
    'Mutation B (MYC Amplification)': network_mut_b,
    'Mutation C (MDM2 Overexpression)': network_mut_c,
    'Mutation D (p21 Knockout)': network_mut_d
}

scenarios = {
    'Healthy Cell': {'DNA_damage': 0, 'p53': 0, 'MYC': 0, 'CDK2': 0, 'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0},
    'Stressed Cell': { 'DNA_damage': 1, 'p53': 0, 'MYC': 0, 'CDK2': 0, 'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0},
    'Oncogene Hijacked Cell': { 'DNA_damage': 0, 'p53': 0, 'MYC': 1, 'CDK2': 0, 'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0}
}


### Scenario Analysis

In [ ]:
# Go through each network
for current_network in networks.items():
    print(f"{current_network[0]}:")
    # Check each scenario
    for current_scenario in scenarios.items():
        
        current_network[1].set_state(**current_scenario[1])
        trajectory = current_network[1].simulate(steps=15) 
        
        result = trajectory[-1]
        node_names = sorted(current_network[1].nodes.keys())
        final_dict = {node: result[i] for i, node in enumerate(node_names)}
        
        print(f"\t{current_scenario[0]}\t: Growth: {final_dict['Growth']} | Death: {final_dict['Death']} | p53: {final_dict['p53']}")


Normal Network:
   Reached steady state after 4 steps
	Healthy Cell	: Growth: 1 | Death: 0 | p53: 0
   Steady state wasn't reached
	Stressed Cell	: Growth: 0 | Death: 1 | p53: 0
   Reached steady state after 3 steps
	Oncogene Hijacked Cell	: Growth: 1 | Death: 0 | p53: 0
Mutation A (p53 Knockout):
   Reached steady state after 4 steps
	Healthy Cell	: Growth: 1 | Death: 0 | p53: 0
   Reached steady state after 4 steps
	Stressed Cell	: Growth: 1 | Death: 0 | p53: 0
   Reached steady state after 3 steps
	Oncogene Hijacked Cell	: Growth: 1 | Death: 0 | p53: 0
Mutation B (MYC Amplification):
   Reached steady state after 4 steps
	Healthy Cell	: Growth: 1 | Death: 0 | p53: 0
   Reached steady state after 7 steps
	Stressed Cell	: Growth: 1 | Death: 0 | p53: 0
   Reached steady state after 3 steps
	Oncogene Hijacked Cell	: Growth: 1 | Death: 0 | p53: 0
Mutation C (MDM2 Overexpression):
   Reached steady state after 4 steps
	Healthy Cell	: Growth: 1 | Death: 0 | p53: 0
   Reached steady state a

### Attractor Analysis

In [ ]:
all_states = list(itertools.product([0, 1], repeat=len(nodes)))
print(f"Testing all {len(all_states)} possible initial states...\n")

# Go through each mutated network
for current_network in networks.items():
    current_network_object = current_network[1] 
    
    unique_attractors = set()
    cancer_states_count = 0  # Track how many initial states lead to cancer

    # Check all states
    for initial_state in all_states:
        state_dict = {nodes[i]: initial_state[i] for i in range(len(nodes))}
        current_network_object.set_state(**state_dict)

        trajectory = current_network_object.simulate(steps=15, to_print=False)

        # Check if reached a steady state
        if len(trajectory) >= 2 and np.array_equal(trajectory[-1], trajectory[-2]):
            final_state = tuple(int(x) for x in trajectory[-1])  # Clean conversion 

            # Check if this is a newly discovered attractor
            if final_state not in unique_attractors:
                unique_attractors.add(final_state)
                
            # Map the final state back to a dictionary to check Growth and Death
            sorted_nodes = sorted(current_network_object.nodes.keys())
            final_dict = {node: final_state[i] for i, node in enumerate(sorted_nodes)}
            
            # Cancer condition: Uncontrolled growth AND no cell death
            if final_dict['Growth'] == 1 and final_dict['Death'] == 0:
                cancer_states_count += 1

    cancer_percentage = (cancer_states_count / len(all_states)) * 100
    
    print(f"{current_network[0]}:")
    print(f"\tUnique steady-state attractors: {len(unique_attractors)}")
    print(f"\tPercentage of states lead to cancer-like states:  {cancer_percentage:5.1f}%")


Testing all 256 possible initial states...

Normal Network:
	Unique steady-state attractors: 3
	Percentage of states lead to cancer-like states:   53.1%
Mutation A (p53 Knockout):
	Unique steady-state attractors: 2
	Percentage of states lead to cancer-like states:  100.0%
Mutation B (MYC Amplification):
	Unique steady-state attractors: 2
	Percentage of states lead to cancer-like states:  100.0%
Mutation C (MDM2 Overexpression):
	Unique steady-state attractors: 2
	Percentage of states lead to cancer-like states:  100.0%
Mutation D (p21 Knockout):
	Unique steady-state attractors: 3
	Percentage of states lead to cancer-like states:   53.1%
